In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report


#Load data

df=pd.read_csv("ml_ready_iot_data.csv")
df.columns=df.columns.str.strip()
df.head()

,device_id,temperature,humidity,usage_hours,error_count,vibration,avg_temp,temp_change,usage_frequency,health_score,health_status,is_risky,device_status
0,D101,57.0,41,24,4,0.44,57.0,0.0,1.000000,58.50,moderate,0,warning
1,D102,32.0,46,22,1,0.93,32.0,0.0,0.916667,76.10,good,0,normal
2,D103,26.6,45,7,3,0.81,26.6,0.0,0.291667,68.92,moderate,0,warning
3,D104,26.3,52,23,6,0.41,26.3,0.0,0.958333,58.01,moderate,1,failure
4,D105,54.5,40,6,6,0.58,54.5,0.0,0.250000,47.85,moderate,1,failure


In [3]:
#Feature & Target
X=df[[
    "temperature","humidity","usage_hours",
    "error_count","vibration","avg_temp",
    "temp_change","usage_frequency","health_score"
]]
y=df["device_status"]

In [4]:
#split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
#Baseline RF
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
rf_pred=rf.predict(X_test)
print("Baseline RF Accurecy:",accuracy_score(y_test,rf_pred))
print("confusion Matrix:",confusion_matrix(y_test,rf_pred))


Baseline RF Accurecy: 0.9
confusion Matrix: [[11  0  0]
 [ 0  0  1]
 [ 1  0  7]]


In [7]:
#hyperparameter tuning

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,                # 3-fold cross validation
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\n Best Parameters:", grid.best_params_)

# Best model
best_rf = grid.best_estimator_

# Predictions
best_pred = best_rf.predict(X_test)

print("\n Tuned RF Accuracy:", accuracy_score(y_test, best_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, best_pred))


 Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

 Tuned RF Accuracy: 0.9
Confusion Matrix:
 [[11  0  0]
 [ 0  0  1]
 [ 1  0  7]]


In [9]:
#comparing with decision tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("\n Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))


 Decision Tree Accuracy: 0.95
